# Qwen3.5-9B · 기존 Windows 로컬 · 전체 학습

기존 AI2_Challenge 프로젝트의 Python 환경·모델·분할·P5 설정을 그대로 사용합니다.
**기존 train+pool 전체(예상 5,214개)**를 원본 모델에서 새로 학습합니다. 기존 검증 500개와 holdout 약 1,000개의 ID·이미지 그룹은 유지하며 학습에서 제외합니다.

| 항목 | 적용 |
|---|---|
| 모델 / 양자화 | Qwen3.5-9B, 기존 revision / NF4 4bit BF16 |
| 학습 / 추론 해상도 | 640² / 672² |
| Loss / 프롬프트 | answer_only / 학습·추론 P5 |
| LR / epoch | 1e-4 / 1 |
| LoRA | rank 8, alpha 16, dropout 0.05 |
| 데이터 | 기존 train+pool 전체, 증강·dev 미사용 |

### 사용
1. 기존 baseline/LR 노트북을 실행한 **AI2_Challenge 프로젝트 폴더**에 이 파일을 놓습니다.
2. 첫 설정 셀의 PROJECT_DIR와 LR_RUN_DIR을 확인합니다. 데이터·모델·baseline 경로는 LR 기록에서 자동으로 가져옵니다.
3. Run All: 기존 환경 확인 → 고정 분할 확장 → 새 학습 → 검증 및 1,000개 기준 비교 → 전체 test → submission.csv.
4. **학습량 실험만 할 때 RUN_TEST=False**로 바꾸면 전체 test와 제출 파일 생성을 생략합니다. 모델·검증 기록은 남습니다.

기존 LoRA는 기준 점수와 설정 확인에만 사용합니다. 이어 학습하지 않습니다. 설치·다운로드·Drive·ZIP 준비는 필요 없습니다.
실제 실행했던 baseline 코드를 해시 확인 후 불러와 Dataset·processor·학습 루프·파싱을 사용합니다.

### 시간
- 새 학습 **1회(예상 5,214개)** + 기존 검증 **500개 1회** + 선택적으로 test **6,714개 1회**.
- 5060 Ti의 1,000개 학습 45분30초를 선형 환산: 학습 약 **3시간57분**, 검증 약 **7분**, 전체 test 약 **1시간37분**.
- RUN_TEST=True: 합계 **약 5시간41분**, False: **약 4시간4분**. 점검·로딩은 별도이며 전체 학습 실측값은 아닙니다.
- 현재 640²/P5 학습+검증 1회 52분43초 대비 약 **6.5배 / 4.6배**입니다. 최초 384² baseline 전체 시간은 없어 그 기준 배수는 미확정입니다.
- 기존 1,000개 검증 결과는 재사용합니다. 새 학습의 저장/재로드 확인 최대 5문항씩 2회와 워밍업도 수행합니다.

학습 도중 중단하면 해당 학습은 처음부터 다시 시작합니다. 완료된 학습·검증은 재사용하고, test는 25문항 단위로 이어갑니다.
결과는 output/TASK-006-local-alltrain의 별도 FULLTRAIN 폴더에 보존합니다. 실제 GPU 전체 학습은 작성 환경에서 실행하지 않았습니다.


## 1. 기존 프로젝트와 LR 결과 경로
기본값은 전달한 LR 결과 폴더입니다. 다른 위치라면 첫 셀에서 수정하세요.

In [1]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,time
PROJECT_DIR = Path.cwd().resolve()  # 기존 AI2_Challenge 프로젝트 루트
LR_RUN_DIR = PROJECT_DIR / "output/TASK-006-lr/LR-6007c4ee02657be3"
ENV_PYTHON = PROJECT_DIR / "downloads/envs/TASK006_baseline_qwen35" / ("Scripts/python.exe" if os.name=="nt" else "bin/python")
SESSION_TAG = "local_alltrain_P5_lr1e4_v1"
RUN_TEST = True  # 학습량 실험만 할 때 False: 전체 test 추론과 submission 생성 생략
if not ENV_PYTHON.is_file():raise FileNotFoundError(f'기존 전용 Python 경로 확인: {ENV_PYTHON}')
for name in ['resolution_config.json','resolution_worker.py','train_lr_1e-4_status.json','res_lr_1e-4_status.json','requirements.lock.txt']:
    if not (LR_RUN_DIR/name).is_file():raise FileNotFoundError(LR_RUN_DIR/name)


## 2. 기존 환경·선택 설정 확인
패키지·코드·학습 파일 해시를 확인합니다. 기존 환경을 변경하거나 패키지를 재설치하지 않습니다.

In [2]:
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def read_json(path):return json.loads(Path(path).read_text(encoding='utf-8'))
freeze=subprocess.check_output([str(ENV_PYTHON),'-m','pip','freeze'],text=True,encoding='utf-8')
if sorted(freeze.splitlines())!=sorted((LR_RUN_DIR/'requirements.lock.txt').read_text(encoding='utf-8').splitlines()):
    raise RuntimeError('LR 실험과 패키지 구성이 다릅니다. 기존 전용 환경을 확인하세요.')
lr_config=read_json(LR_RUN_DIR/'resolution_config.json')
BASELINE_RUN_DIR=Path(lr_config['baseline_dir'])
base_config=read_json(BASELINE_RUN_DIR/'run_config.json')
if file_hash(BASELINE_RUN_DIR/'run_config.json')!=lr_config['baseline_config_sha256']:raise RuntimeError('baseline 설정 변경')
raw=(LR_RUN_DIR/'resolution_worker.py').read_bytes()
if lr_config['worker_sha256'] not in [hashlib.sha256(raw).hexdigest(),hashlib.sha256(raw.replace(b'\r\n',b'\n')).hexdigest()]:raise RuntimeError('LR 코드 변경')
train_record=read_json(LR_RUN_DIR/'train_lr_1e-4_status.json')
if train_record['state']!='completed':raise RuntimeError('기준 학습 미완료')
for name,h in train_record['artifacts'].items():
    if file_hash(Path(train_record['directory'])/name)!=h:raise RuntimeError(f'기준 학습 파일 변경: {name}')
selected=read_json(Path(train_record['directory'])/'training_config.json')
for key,value in {'model_id':'Qwen/Qwen3.5-9B','pixel_budget':640**2,'learning_rate':1e-4,'loss_mode':'answer_only','prompt':'P5','epochs':1}.items():
    if selected.get(key)!=value:raise RuntimeError(f'선택 모델 설정 불일치: {key}')
if selected['p5_instruction']!=lr_config['p5_instruction']:raise RuntimeError('P5 불일치')
audit_record=read_json(BASELINE_RUN_DIR/'audit_complete.json')
SPLIT_MANIFEST=Path(audit_record['directory'])/'split_manifest.csv'
DATA_DIR=Path(base_config['data_dir']);MODEL_DIR=Path(base_config['model_dir'])
print('기존 baseline:',BASELINE_RUN_DIR)
print('데이터:',DATA_DIR)
print('다운로드된 모델:',MODEL_DIR)
print('그대로 유지할 분할:',SPLIT_MANIFEST)


기존 baseline: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c
데이터: C:\Users\SSAFY\Desktop\AI2_Challenge\data
다운로드된 모델: C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\models\Qwen3_5_9B\c202236235762e1c871ad0ccb60c8ee5ba337b9a
그대로 유지할 분할: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c\audit_20260922_114941_873f43ca\split_manifest.csv


## 3. 로컬 데이터·모델 경로 확인
기존 data/train.csv, data/test.csv, data/sample_submission.csv 및 이미지·모델 파일을 사용합니다.

In [3]:
for name in ['train.csv','test.csv','sample_submission.csv']:
    if not (DATA_DIR/name).is_file():raise FileNotFoundError(DATA_DIR/name)
if not MODEL_DIR.is_dir():raise FileNotFoundError(MODEL_DIR)
print('로컬 입력 파일 확인 완료. 별도 ZIP 압축 해제·설치·모델 다운로드 없음.')


로컬 입력 파일 확인 완료. 별도 ZIP 압축 해제·설치·모델 다운로드 없음.


## 4. 전체 학습 실행 코드
각 단계는 기존 전용 Python의 별도 프로세스에서 실행합니다.

In [4]:
WORKER_SOURCE = '# Original baseline structure: Dataset -> Collator -> DataLoader -> AdamW training.\nimport argparse, csv, gc, hashlib, json, math, os, random, re, sys, time, traceback\nfrom pathlib import Path\nfrom dataclasses import dataclass\nfrom typing import Any\n\n\ndef write_json(path,obj):\n    path=Path(path); tmp=path.with_name(path.name+\'.tmp\')\n    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding=\'utf-8\'); tmp.replace(path)\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for block in iter(lambda:f.read(4*1024*1024),b\'\'): h.update(block)\n    return h.hexdigest()\n\n\ndef block_network():\n    os.environ.update(HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\',HF_HUB_DISABLE_TELEMETRY=\'1\',\n        WANDB_DISABLED=\'true\',TOKENIZERS_PARALLELISM=\'false\',CUBLAS_WORKSPACE_CONFIG=\':4096:8\')\n    def guard(event,args):\n        if event in {\'socket.connect\',\'socket.getaddrinfo\',\'socket.sendto\'}: raise RuntimeError(\'로컬 실행 중 네트워크 접근 금지\')\n    sys.addaudithook(guard)\n\n\ndef read_table(path,columns):\n    import pandas as pd\n    df=pd.read_csv(path,dtype=str,keep_default_na=False)\n    if set(columns)-set(df.columns): raise ValueError(\'필수 컬럼 누락\')\n    if df.empty or df.id.duplicated().any(): raise ValueError(\'빈 데이터 또는 중복 ID\')\n    for c in columns:\n        if df[c].str.strip().eq(\'\').any(): raise ValueError(f\'빈 필수 항목: {c}\')\n    return df\n\n\ndef resolve_image(root,value):\n    value=str(value).replace(\'\\\\\',\'/\')\n    root=Path(root).resolve(); p=(root/value).resolve()\n    if \'://\' in value or \':\' in value or not p.is_relative_to(root) or not p.is_file(): raise FileNotFoundError(p)\n    return p\n\n\nSYSTEM_INSTRUCT = (\n    "You are a helpful visual question answering assistant. "\n    "Answer using exactly one letter among a, b, c, or d. No explanation."\n)\n\n\ndef build_mc_prompt(question,a,b,c,d):\n    return (f\'{question}\\n(a) {a}\\n(b) {b}\\n(c) {c}\\n(d) {d}\\n\\n\'\n            \'정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.\')\n\n\ndef parse_answer(text):\n    # Baseline\'s implicit \'a\' replaced by explicit failure. No fallback in this baseline.\n    s=re.sub(r\'^(?:answer|정답)\\s*[:：]\\s*\',\'\',str(text).strip(),flags=re.I)\n    m=re.fullmatch(r\'(?:\\(([a-d])\\)|([a-d]))[.。]?\',s,flags=re.I)\n    return (m.group(1) or m.group(2)).lower() if m else None\n\n\nfrom torch.utils.data import Dataset\n\n\nclass VQAMCDataset(Dataset):\n    def __init__(self,df,processor,train=True,data_dir=None):\n        self.df=df.reset_index(drop=True); self.processor=processor; self.train=train; self.data_dir=data_dir\n    def __len__(self): return len(self.df)\n    def __getitem__(self,i):\n        from PIL import Image,ImageOps\n        row=self.df.iloc[i]\n        with Image.open(resolve_image(self.data_dir,row[\'path\'])) as f:\n            img=ImageOps.exif_transpose(f).convert(\'RGB\')\n        user_text=build_mc_prompt(*(str(row[k]) for k in [\'question\',\'a\',\'b\',\'c\',\'d\']))\n        messages=[{\'role\':\'system\',\'content\':[{\'type\':\'text\',\'text\':SYSTEM_INSTRUCT}]},\n                  {\'role\':\'user\',\'content\':[{\'type\':\'image\',\'image\':img},{\'type\':\'text\',\'text\':user_text}]}]\n        if self.train:\n            messages.append({\'role\':\'assistant\',\'content\':[{\'type\':\'text\',\'text\':str(row[\'answer\'])}]})\n        return {\'messages\':messages,\'image\':img}\n\n\n@dataclass\nclass DataCollator:\n    processor: Any\n    train: bool=True\n    max_input_tokens: int=4096\n    def __call__(self,batch):\n        texts,images=[],[]\n        for sample in batch:\n            texts.append(self.processor.apply_chat_template(sample[\'messages\'],tokenize=False,\n                add_generation_prompt=not self.train,enable_thinking=False))\n            images.append(sample[\'image\'])\n        enc=self.processor(text=texts,images=images,padding=True,return_tensors=\'pt\',add_special_tokens=False)\n        if enc[\'input_ids\'].shape[1]>self.max_input_tokens: raise ValueError(\'입력 길이 초과: 자동 잘라내기 없음\')\n        if self.train:\n            # Original baseline full-sequence labels retained, including media/role tokens.\n            # Only artificial padding is ignored. Batch size 1 normally has no padding.\n            enc[\'labels\']=enc[\'input_ids\'].clone()\n            enc[\'labels\'][enc[\'attention_mask\']==0]=-100\n        return enc\n\n\nclass ModelAdapter:\n    """Stage runner bridge; actual input/training uses baseline Dataset and Collator."""\n    def __init__(self,cfg,out):\n        import torch\n        from transformers import AutoProcessor,AutoModelForImageTextToText,BitsAndBytesConfig\n        self.cfg=cfg;self.out=out; self.dtype=torch.bfloat16;self.device=\'cuda:0\'\n        if not torch.cuda.is_bf16_supported(): raise RuntimeError(\'BF16 지원 GPU 필요\')\n        self.processor=AutoProcessor.from_pretrained(cfg[\'model_dir\'],local_files_only=True)\n        ip=self.processor.image_processor; pixels=cfg[\'pixel_budget\']\n        ip.size={\'shortest_edge\':pixels,\'longest_edge\':pixels}\n        if hasattr(ip,\'min_pixels\'): ip.min_pixels=pixels\n        if hasattr(ip,\'max_pixels\'): ip.max_pixels=pixels\n        self.tokenizer=self.processor.tokenizer\n        bnb_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_use_double_quant=True,\n            bnb_4bit_quant_type=\'nf4\',bnb_4bit_compute_dtype=self.dtype)\n        self.model=AutoModelForImageTextToText.from_pretrained(cfg[\'model_dir\'],\n            local_files_only=True,quantization_config=bnb_config,device_map={\'\':\'cuda:0\'},\n            dtype=self.dtype,attn_implementation=\'sdpa\')\n        self.model.config.use_cache=False; self.model.eval()\n        self.processor.save_pretrained(out/\'processor\')\n        write_json(out/\'model_config.json\',self.model.config.to_dict())\n        write_json(out/\'processor_settings.json\',ip.to_dict())\n    def add_lora(self):\n        import torch\n        from peft import LoraConfig,get_peft_model\n        # Memory-safe k-bit preparation: avoid full embedding FP32 expansion on 16 GB.\n        for name,p in self.model.named_parameters():\n            p.requires_grad_(False)\n            if \'norm\' in name.lower() and p.ndim==1 and p.is_floating_point(): p.data=p.data.to(torch.float32)\n        self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n        self.model.enable_input_require_grads()\n        lora_config=LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,bias=\'none\',\n            target_modules=[\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\',\'gate_proj\',\'up_proj\',\'down_proj\'],\n            task_type=\'CAUSAL_LM\')\n        self.model=get_peft_model(self.model,lora_config)\n        self.model.print_trainable_parameters()\n        write_json(self.out/\'lora_parameters.json\',[{\'name\':n,\'shape\':list(p.shape)} for n,p in self.model.named_parameters() if p.requires_grad])\n    def move(self,batch):\n        return {k:v.to(self.device,dtype=self.dtype if v.is_floating_point() else v.dtype) for k,v in batch.items()}\n    def encode(self,row,training=False):\n        import pandas as pd\n        ds=VQAMCDataset(pd.DataFrame([row]),self.processor,training,self.cfg[\'data_dir\'])\n        sample=ds[0]; enc=DataCollator(self.processor,training,self.cfg[\'max_input_tokens\'])([sample])\n        grid=enc.get(\'image_grid_thw\'); gs=grid.tolist() if grid is not None else []\n        ip=self.processor.image_processor; patch=getattr(ip,\'patch_size\',16); merge=getattr(ip,\'merge_size\',2)\n        self.last_image_meta={\'original_size\':list(sample[\'image\'].size),\'grid_thw\':gs,\n            \'processed_hw\':[[int(g[1])*patch,int(g[2])*patch] for g in gs],\n            \'visual_tokens\':sum(math.prod(g)//(merge*merge) for g in gs)}\n        return self.move(enc)\n    def loss(self,inputs): return self.model(**inputs,use_cache=False).loss\n    def generate(self,inputs):\n        import torch\n        self.model.eval()\n        with torch.inference_mode(),torch.autocast(\'cuda\',dtype=self.dtype):\n            ids=self.model.generate(**inputs,max_new_tokens=self.cfg[\'max_new_tokens\'],do_sample=False,\n                use_cache=True,eos_token_id=self.tokenizer.eos_token_id)\n        # Original baseline decoded the whole input; now decode generated tokens only.\n        return self.tokenizer.decode(ids[0,inputs[\'input_ids\'].shape[1]:],skip_special_tokens=True).strip()\n    def reload_adapter(self,path):\n        from peft import PeftModel\n        base=self.model.unload()\n        self.model=PeftModel.from_pretrained(base,str(path),local_files_only=True,is_trainable=False)\n        self.model.eval()\n\n\ndef evaluate(adapter,df,label,out):\n    import torch,pandas as pd\n    adapter.model.eval();torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();start=time.perf_counter()\n    rows=[]\n    fields=[\'id\',\'answer\',\'raw_output\',\'parse_failed\',\'fallback_used\',\'gold\',\'correct\',\'strict_correct\',\'group_id\',\'question_type\',\'image_meta\']\n    with open(out/f\'{label}_predictions.csv\',\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()\n        for i,row in enumerate(df.to_dict(\'records\')):\n            inputs=adapter.encode({k:v for k,v in row.items() if k!=\'answer\'})\n            raw=adapter.generate(inputs);answer=parse_answer(raw)\n            rec={\'id\':row[\'id\'],\'answer\':answer or \'\', \'raw_output\':raw,\'parse_failed\':answer is None,\n                 \'fallback_used\':False,\'gold\':row[\'answer\'],\'correct\':answer==row[\'answer\'],\n                 \'strict_correct\':answer==row[\'answer\'],\'group_id\':row[\'group_id\'],\'question_type\':row[\'question_type\'],\n                 \'image_meta\':json.dumps(adapter.last_image_meta)}\n            writer.writerow(rec);f.flush();rows.append(rec);del inputs\n            if (i+1)%25==0 or i+1==len(df): print(label,i+1,\'/\',len(df),flush=True)\n    torch.cuda.synchronize();seconds=time.perf_counter()-start;n=len(rows)\n    metrics={\'n\':n,\'correct_n\':sum(r[\'correct\'] for r in rows),\'accuracy\':sum(r[\'correct\'] for r in rows)/n,\n        \'parse_failure_rate\':sum(r[\'parse_failed\'] for r in rows)/n,\'fallback_usage_rate\':0.,\n        \'seconds\':seconds,\'seconds_per_sample\':seconds/n,\'peak_allocated_gib\':torch.cuda.max_memory_allocated()/2**30,\n        \'peak_reserved_gib\':torch.cuda.max_memory_reserved()/2**30}\n    write_json(out/f\'{label}_metrics.json\',metrics);print(metrics,flush=True)\n    return rows,metrics\n\n\ndef train_one_epoch(adapter,df,cfg,out):\n    import torch,pandas as pd\n    from torch.utils.data import DataLoader\n    from transformers import get_linear_schedule_with_warmup\n    # Baseline DataLoader, AdamW, warmup, accumulation, one epoch retained.\n    train_ds=VQAMCDataset(df,adapter.processor,train=True,data_dir=cfg[\'data_dir\'])\n    train_loader=DataLoader(train_ds,batch_size=1,shuffle=True,\n        generator=torch.Generator().manual_seed(cfg[\'seed\']),\n        collate_fn=DataCollator(adapter.processor,True,cfg[\'max_input_tokens\']),num_workers=0)\n    model=adapter.model; GRAD_ACCUM=cfg[\'gradient_accumulation\']\n    params=[p for p in model.parameters() if p.requires_grad]\n    optimizer=torch.optim.AdamW(params,lr=cfg[\'learning_rate\'])\n    num_training_steps=math.ceil(len(train_loader)/GRAD_ACCUM)\n    scheduler=get_linear_schedule_with_warmup(optimizer,int(num_training_steps*.03),num_training_steps)\n    # BF16 needs no GradScaler; original mixed FP16 compute/BF16 autocast is unified.\n    model.train(); optimizer.zero_grad(set_to_none=True)\n    torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();started=time.perf_counter();rows=[];running=0.\n    for step,batch in enumerate(train_loader,start=1):\n        batch=adapter.move(batch)\n        # Correct denominator and optimizer step for the final incomplete group.\n        group_start=((step-1)//GRAD_ACCUM)*GRAD_ACCUM\n        group_size=min(GRAD_ACCUM,len(train_loader)-group_start)\n        with torch.autocast(\'cuda\',dtype=adapter.dtype):\n            outputs=model(**batch,use_cache=False);loss=outputs.loss\n        if not torch.isfinite(loss): raise FloatingPointError(\'loss NaN/Inf\')\n        raw_loss=float(loss.detach());(loss/group_size).backward();running+=raw_loss\n        del outputs,loss,batch\n        if step%GRAD_ACCUM==0 or step==len(train_loader):\n            grads=[p.grad for p in params if p.grad is not None]\n            if not grads or not all(bool(torch.isfinite(g).all()) for g in grads): raise FloatingPointError(\'gradient 오류\')\n            optimizer.step();optimizer.zero_grad(set_to_none=True);scheduler.step()\n            rec={\'update\':len(rows)+1,\'mean_loss\':running/group_size,\'lr\':scheduler.get_last_lr()[0]};rows.append(rec);running=0.\n            with open(out/\'train_updates.jsonl\',\'a\',encoding=\'utf-8\') as f:f.write(json.dumps(rec)+\'\\n\')\n            print(\'train\',rec[\'update\'],\'/\',num_training_steps,rec,flush=True)\n    torch.cuda.synchronize()\n    metrics={\'train_n\':len(df),\'epochs\':1,\'updates\':len(rows),\'seconds\':time.perf_counter()-started,\n        \'peak_allocated_gib\':torch.cuda.max_memory_allocated()/2**30,\'peak_reserved_gib\':torch.cuda.max_memory_reserved()/2**30}\n    pd.DataFrame(rows).to_csv(out/\'train_log.csv\',index=False);write_json(out/\'train_metrics.json\',metrics)\n    del optimizer,scheduler,params,grads,train_loader;gc.collect();torch.cuda.empty_cache()\n    return metrics\n\ndef question_type(text):\n    # Heuristic diagnostic tags, not verified semantic labels.\n    if re.search(r\'아닌|않은|않는|없는|제외\', text): return \'negation\'\n    if re.search(r\'가격|얼마|원인가|금액|할인\', text): return \'price\'\n    if re.search(r\'왼쪽|오른쪽|위쪽|아래|옆|위치\', text): return \'position\'\n    if re.search(r\'몇|숫자|날짜|시간|번호|수량\', text): return \'number\'\n    return \'text_other\'\n\n\ndef take_groups(frame, target, seed):\n    import numpy as np\n    # Never split a group. Greedy size selection over repeated seeded permutations;\n    # use answer/type proportions to break ties and minimize distribution shift.\n    groups = frame.groupby(\'group_id\', sort=True).size()\n    if target >= len(frame): return set(groups.index)\n    rng = np.random.default_rng(seed)\n    best, best_score = None, float(\'inf\')\n    answer_ref = frame.answer.value_counts(normalize=True)\n    type_ref = frame.question_type.value_counts(normalize=True)\n    for _ in range(100):\n        chosen, n = [], 0\n        for g in rng.permutation(groups.index.to_numpy()):\n            size = int(groups[g])\n            if abs(n + size - target) < abs(n-target):\n                chosen.append(g); n += size\n        sub = frame[frame.group_id.isin(chosen)]\n        if sub.empty: continue\n        def distance(col, ref):\n            return (sub[col].value_counts(normalize=True).reindex(ref.index,fill_value=0)-ref).abs().sum()\n        score = abs(n-target) + .1 * (distance(\'answer\',answer_ref)+distance(\'question_type\',type_ref))\n        if score < best_score: best, best_score = set(chosen), score\n    if best is None: raise ValueError(\'그룹 크기로 인해 분할 불가\')\n    return best\n\n\ndef audit(cfg, out):\n    import pandas as pd\n    import numpy as np\n    from PIL import Image, ImageOps\n    root = Path(cfg[\'data_dir\'])\n    df = read_table(root/\'train.csv\', [\'id\',\'path\',\'question\',\'a\',\'b\',\'c\',\'d\',\'answer\']).sort_values(\'id\').reset_index(drop=True)\n    if not df.answer.isin(list(\'abcd\')).all(): raise ValueError(\'라벨은 소문자 a~d여야 합니다.\')\n    findings = []\n    for row in df.to_dict(\'records\'):\n        for col in [\'question\',\'a\',\'b\',\'c\',\'d\']:\n            if row[col] != row[col].strip(): findings.append({\'id\':row[\'id\'],\'field\':col,\'reason\':\'leading/trailing whitespace\',\'action\':\'report only\'})\n            if \'\\ufffd\' in row[col]: findings.append({\'id\':row[\'id\'],\'field\':col,\'reason\':\'replacement character\',\'action\':\'review\'})\n        if len({row[c].strip() for c in \'abcd\'}) < 4:\n            findings.append({\'id\':row[\'id\'],\'field\':\'options\',\'reason\':\'duplicate options\',\'action\':\'review\'})\n    pd.DataFrame(findings,columns=[\'id\',\'field\',\'reason\',\'action\']).to_csv(out/\'data_findings.csv\',index=False)\n    images, errors = [], []\n    for i,row in enumerate(df.to_dict(\'records\')):\n        try:\n            p = resolve_image(root,row[\'path\'])\n            with Image.open(p) as raw:\n                raw.load()\n                orientation = raw.getexif().get(274,1)\n                im = ImageOps.exif_transpose(raw).convert(\'RGB\')\n                pixel_hash = hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()\n                tiny=np.asarray(im.convert(\'L\').resize((9,8),Image.Resampling.LANCZOS))\n                bits=(tiny[:,1:]>tiny[:,:-1]).reshape(-1)\n                dh=sum(int(v)<<j for j,v in enumerate(bits))\n                stat=p.stat()\n                images.append(dict(id=row[\'id\'],path=row[\'path\'],width=im.width,height=im.height,\n                    orientation=orientation,pixel_sha256=pixel_hash,dhash=f\'{dh:016x}\',\n                    file_sha256=sha256(p),bytes=stat.st_size,mtime_ns=stat.st_mtime_ns))\n        except Exception as exc: errors.append(dict(id=row[\'id\'],path=row[\'path\'],error=str(exc)))\n        if (i+1)%250==0: print(\'image audit\',i+1,\'/\',len(df),flush=True)\n    pd.DataFrame(errors,columns=[\'id\',\'path\',\'error\']).to_csv(out/\'image_errors.csv\',index=False)\n    if errors: raise RuntimeError(f\'누락/손상 이미지 {len(errors)}개. image_errors.csv 확인. 자동 제외하지 않습니다.\')\n    parent=list(range(len(images)))\n    def find(i):\n        while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]\n        return i\n    def union(i,j):\n        a,b=find(i),find(j)\n        if a!=b: parent[max(a,b)]=min(a,b)\n    exact={}; candidates=[]; hashes=[int(x[\'dhash\'],16) for x in images]\n    for i,row in enumerate(images):\n        h=row[\'pixel_sha256\']\n        if h in exact: union(i,exact[h])\n        else: exact[h]=i\n        ratio=row[\'width\']/row[\'height\']\n        for j in range(i):\n            if images[j][\'pixel_sha256\']==h: continue\n            distance=(hashes[i]^hashes[j]).bit_count()\n            if distance<=cfg[\'near_hash_distance\'] and abs(math.log(ratio/(images[j][\'width\']/images[j][\'height\'])))<=.05:\n                # Conservative candidate grouping; false positives can reduce train pool.\n                union(i,j)\n                candidates.append({\'id_a\':images[j][\'id\'],\'id_b\':row[\'id\'],\'distance\':distance,\n                                   \'policy\':\'same group conservatively; not human verified\'})\n    df[\'group_id\']=[\'g_\'+images[find(i)][\'id\'] for i in range(len(images))]\n    df[\'question_type\']=df.question.map(question_type)\n    for i,row in enumerate(images): row[\'group_id\']=df.iloc[i].group_id\n    pd.DataFrame(images).to_csv(out/\'image_audit.csv\',index=False)\n    pd.DataFrame(candidates,columns=[\'id_a\',\'id_b\',\'distance\',\'policy\']).to_csv(out/\'near_duplicate_candidates.csv\',index=False)\n    if cfg[\'valid_n\']+cfg[\'holdout_n\']>=len(df):\n        raise ValueError(\'데이터 개수보다 분할 요청이 큽니다. config 값을 확인하세요.\')\n    final=take_groups(df,cfg[\'holdout_n\'],cfg[\'seed\'])\n    remaining=df[~df.group_id.isin(final)]\n    valid=take_groups(remaining,cfg[\'valid_n\'],cfg[\'seed\']+1)\n    remaining=remaining[~remaining.group_id.isin(valid)]\n    train=set(remaining.group_id)  # Every group outside valid and holdout.\n    df[\'split\']=[\'holdout\' if g in final else \'valid\' if g in valid else \'train\' if g in train else \'pool\' for g in df.group_id]\n    if not set([\'train\',\'valid\',\'holdout\']).issubset(set(df.split)): raise ValueError(\'빈 분할이 있습니다.\')\n    assert df.groupby(\'group_id\').split.nunique().max()==1\n    split=df[[\'id\',\'split\',\'group_id\',\'question_type\']]\n    split.to_csv(out/\'split_manifest.csv\',index=False)\n    for name in [\'train\',\'valid\',\'holdout\',\'pool\']:\n        split[split.split==name].to_csv(out/f\'{name}_ids.csv\',index=False)\n    dist=pd.crosstab(df.split,df.answer).reindex(columns=list(\'abcd\'),fill_value=0)\n    dist.to_csv(out/\'answer_distribution.csv\')\n    pd.crosstab(df.split,df.question_type).to_csv(out/\'type_distribution.csv\')\n    manifest={\'csv_sha256\':sha256(root/\'train.csv\'),\'split_sha256\':sha256(out/\'split_manifest.csv\'),\n              \'image_audit_sha256\':sha256(out/\'image_audit.csv\'),\'counts\':df.split.value_counts().to_dict(),\n              \'groups\':int(df.group_id.nunique()),\'near_candidate_pairs\':len(candidates),\n              \'near_policy\':\'dHash candidate transitive grouping; human review pending; misses possible\',\n              \'question_type_policy\':\'regex heuristic only\',\'review_02\':\'pending\',\'dev_used\':False,\n              \'test_used\':False,\'findings\':len(findings)}\n    write_json(out/\'data_manifest.json\',manifest)\n    print(dist.to_string(),flush=True); print(json.dumps(manifest,ensure_ascii=False,indent=2),flush=True)\n    return manifest\n\n\ndef completed(root, stage):\n    p=root/f\'{stage}_complete.json\'\n    if not p.exists(): return None\n    rec=json.loads(p.read_text(encoding=\'utf-8\'))\n    for name,digest in rec[\'artifacts\'].items():\n        path=Path(rec[\'directory\'])/name\n        if not path.is_file() or sha256(path)!=digest: raise RuntimeError(f\'완료 결과가 수정/누락되었습니다: {path}\')\n    return rec\n\n\ndef load_split(cfg, root):\n    import pandas as pd\n    record=completed(root,\'audit\')\n    if record is None: raise RuntimeError(\'audit 단계를 먼저 실행하세요.\')\n    path=Path(record[\'directory\'])\n    meta=json.loads((path/\'data_manifest.json\').read_text(encoding=\'utf-8\'))\n    if sha256(Path(cfg[\'data_dir\'])/\'train.csv\')!=meta[\'csv_sha256\']: raise RuntimeError(\'train.csv 변경: 새 SESSION_TAG로 분할부터 실행하세요.\')\n    # Validate image bytes: changed data cannot silently reuse completed experiments.\n    images=pd.read_csv(path/\'image_audit.csv\',dtype=str)\n    for r in images.to_dict(\'records\'):\n        p=resolve_image(cfg[\'data_dir\'],r[\'path\'])\n        if sha256(p)!=r[\'file_sha256\']: raise RuntimeError(f"이미지 변경: {r[\'id\']}. 새 SESSION_TAG 필요")\n    df=pd.read_csv(Path(cfg[\'data_dir\'])/\'train.csv\',dtype=str,keep_default_na=False)\n    split=pd.read_csv(path/\'split_manifest.csv\',dtype=str)\n    merged=df.merge(split,on=\'id\',validate=\'one_to_one\')\n    assert len(merged)==len(df) and merged.groupby(\'group_id\').split.nunique().max()==1\n    return merged[merged.split==\'train\'].copy(),merged[merged.split==\'valid\'].copy(),meta\n\n\ndef gpu_environment(cfg,out):\n    import torch, numpy as np, platform\n    from importlib.metadata import version\n    if not torch.cuda.is_available(): raise RuntimeError(\'CUDA GPU를 찾을 수 없습니다. 드라이버/설치를 확인하세요.\')\n    random.seed(cfg[\'seed\']); np.random.seed(cfg[\'seed\']); torch.manual_seed(cfg[\'seed\']); torch.cuda.manual_seed_all(cfg[\'seed\'])\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.use_deterministic_algorithms(True,warn_only=True)\n    x=torch.ones((32,32),device=\'cuda\',dtype=torch.bfloat16)\n    assert float((x@x)[0,0])==32.; del x\n    info={\'python\':sys.version,\'os\':platform.platform(),\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\n          \'gpu\':torch.cuda.get_device_name(),\'vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n          \'packages\':{p:version(p) for p in [\'transformers\',\'peft\',\'bitsandbytes\',\'accelerate\']},\n          \'network\':\'blocked; local model files only\'}\n    write_json(out/\'environment.json\',info); print(info,flush=True)\n\n\ndef answer_span(full,empty,eos):\n    start=0\n    while start<min(len(full),len(empty)) and full[start]==empty[start]:start+=1\n    suffix=0\n    while suffix<min(len(full)-start,len(empty)-start) and full[-1-suffix]==empty[-1-suffix]:suffix+=1\n    end=len(full)-suffix\n    if start>=end:raise RuntimeError(\'정답 token span을 찾지 못했습니다.\')\n    if end>=len(full) or full[end]!=eos:\n        raise RuntimeError(\'정답 바로 뒤 종료 토큰이 예상과 다릅니다. 템플릿 검토 필요; 자동 우회 없음.\')\n    return start,end\n\n\ndef install_answer_collator(module):\n    import copy\n    original=module.DataCollator\n    class AnswerOnlyCollator(original):\n        def __call__(self,batch):\n            enc=super().__call__(batch)\n            if not self.train:return enc\n            if len(batch)!=1:raise ValueError(\'이 실험은 baseline과 동일하게 batch_size=1\')\n            empty=[]\n            for sample in batch:\n                msgs=copy.deepcopy(sample[\'messages\'])\n                if msgs[-1][\'role\']!=\'assistant\':raise RuntimeError(\'assistant 정답 없음\')\n                gold=msgs[-1][\'content\'][0][\'text\']\n                msgs[-1][\'content\']=[{\'type\':\'text\',\'text\':\'\'}]\n                empty.append({\'messages\':msgs,\'image\':sample[\'image\']})\n            empty_enc=super().__call__(empty)\n            full=enc[\'input_ids\'][0].tolist();blank=empty_enc[\'input_ids\'][0].tolist()\n            start,end=answer_span(full,blank,self.processor.tokenizer.eos_token_id)\n            decoded=self.processor.tokenizer.decode(full[start:end],skip_special_tokens=False).strip()\n            if decoded!=gold:raise RuntimeError(f\'정답 토큰 검증 실패: {decoded!r} != {gold!r}\')\n            enc[\'labels\'].fill_(-100)\n            enc[\'labels\'][0,start:end+1]=enc[\'input_ids\'][0,start:end+1]\n            return enc\n    module.DataCollator=AnswerOnlyCollator\n\n\ndef audit_targets(module,adapter,train,out):\n    import pandas as pd\n    rows=[]\n    for row in train.head(3).to_dict(\'records\'):\n        enc=adapter.encode(row,training=True)\n        ids=enc[\'input_ids\'][0].tolist();labels=enc[\'labels\'][0].tolist()\n        for i,t in enumerate(ids):\n            rows.append({\'id\':row[\'id\'],\'position\':i,\'token_id\':t,\n                         \'token\':adapter.tokenizer.convert_ids_to_tokens(t),\'supervised\':labels[i]!=-100})\n        del enc\n    pd.DataFrame(rows).to_csv(out/\'supervised_tokens.csv\',index=False,encoding=\'utf-8-sig\')\n\n\n\nimport os,sys,json,csv,hashlib,importlib.util,time,uuid,contextlib\nfrom pathlib import Path\n\ndef read(p):return json.loads(Path(p).read_text(encoding=\'utf-8\'))\ndef write(p,obj):\n    p=Path(p);tmp=p.with_name(p.name+\'.tmp\');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\');tmp.replace(p)\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef tables(cfg):\n    import pandas as pd\n    test=pd.read_csv(cfg[\'test_csv\'],dtype=str,keep_default_na=False)\n    sample=pd.read_csv(cfg[\'sample_csv\'],dtype=str,keep_default_na=False)\n    cols=[\'id\',\'path\',\'question\',\'a\',\'b\',\'c\',\'d\']\n    if not set(cols)<=set(test.columns) or list(sample.columns)!=[\'id\',\'answer\']:raise ValueError(\'test/sample_submission 컬럼 확인 필요\')\n    if test.empty or len(test)!=len(sample):raise ValueError(\'test/sample 행 수 불일치\')\n    for name,d in [(\'test\',test),(\'sample\',sample)]:\n        if d.id.duplicated().any() or d.id.str.strip().eq(\'\').any():raise ValueError(f\'{name} ID 중복/누락\')\n    if set(test.id)!=set(sample.id):raise ValueError(\'test/sample ID 집합 불일치\')\n    for c in cols:\n        if test[c].str.strip().eq(\'\').any():raise ValueError(f\'test 빈 값: {c}\')\n    # Sample IDs are authoritative ordering; no question/choice text modifications.\n    test=test.set_index(\'id\').loc[sample.id].reset_index()[cols]\n    return test,sample\n\n\ndef load_progress(root,fingerprint,ids):\n    p=root/\'progress.json\'\n    progress=read(p) if p.exists() else {\'fingerprint\':fingerprint,\'chunks\':[]}\n    if progress[\'fingerprint\']!=fingerprint:raise RuntimeError(\'재개 설정 불일치\')\n    records=[]\n    for chunk in progress[\'chunks\']:\n        path=root/chunk[\'file\']\n        if sha(path)!=chunk[\'sha256\']:raise RuntimeError(\'부분 예측 파일 변경\')\n        records.extend(read(path))\n    if [r[\'id\'] for r in records]!=ids[:len(records)] or len(records)>len(ids):raise RuntimeError(\'부분 예측 ID 중복/순서 불일치\')\n    return progress,records\n\ndef save_chunk(root,progress,rows):\n    if not rows:return\n    name=f\'chunks/part_{len(progress["chunks"]):05d}.json\';p=root/name;p.parent.mkdir(exist_ok=True)\n    write(p,rows);progress[\'chunks\'].append({\'file\':name,\'sha256\':sha(p)})\n    write(root/\'progress.json\',progress)\n\n\ndef export(cfg):\n    import pandas as pd\n    root=Path(cfg[\'output_dir\']);test,sample=tables(cfg);f=read(root/\'frozen_inputs.json\')\n    if f[\'fingerprint\']!=cfg[\'fingerprint\'] or sha(cfg[\'test_csv\'])!=f[\'test_sha256\'] or sha(cfg[\'sample_csv\'])!=f[\'sample_sha256\']:raise RuntimeError(\'입력 CSV 변경\')\n    _,records=load_progress(root,cfg[\'fingerprint\'],test.id.tolist())\n    if len(records)!=len(test):raise RuntimeError(f\'추론 미완료: {len(records)}/{len(test)}\')\n    preds=pd.DataFrame(records);preds.assign(image_meta=preds.image_meta.map(json.dumps)).to_csv(root/\'test_predictions.csv\',index=False,encoding=\'utf-8-sig\')\n    failures=preds[~preds.answer.isin(list(\'abcd\'))];failures.to_csv(root/\'parse_failures.csv\',index=False,encoding=\'utf-8-sig\')\n    report={\'n\':len(test),\'parse_failure_n\':len(failures),\'fallback_n\':int(preds.fallback_used.sum()),\'id_order_matches_sample\':preds.id.tolist()==sample.id.tolist(),\n            \'checkpoint\':f[\'checkpoint\'],\'auto_submitted\':False,\'review_02\':\'pending\',\'test_accuracy\':None}\n    if len(failures) or report[\'fallback_n\'] or not report[\'id_order_matches_sample\']:\n        report[\'state\']=\'blocked\';write(root/\'submission_validation.json\',report)\n        raise RuntimeError(\'파싱/ID 검사 실패. parse_failures.csv 확인. 임의 답 채우기 없이 submission 생성 중단\')\n    result=sample.copy();result[\'answer\']=preds.answer.tolist()\n    tmp=root/\'submission.csv.tmp\';result.to_csv(tmp,index=False,encoding=\'utf-8\')\n    check=pd.read_csv(tmp,dtype=str,keep_default_na=False)\n    if list(check.columns)!=list(sample.columns) or not check.equals(result):raise RuntimeError(\'CSV 재로드 검사 실패\')\n    final=root/\'submission.csv\'\n    if final.exists() and sha(final)!=sha(tmp):raise RuntimeError(\'기존 submission 내용과 다릅니다. 기존 파일 보존 후 확인하세요.\')\n    tmp.replace(final);report.update(state=\'file_checks_passed\',sha256=sha(final),path=str(final),answer_counts=check.answer.value_counts().to_dict())\n    write(root/\'submission_validation.json\',report)\n    (root/\'PROJECT_STATUS_update.md\').write_text(\'# TASK-006 전체 test 추론\\n\\n\'+json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\\n제출 CSV 생성 및 파일 검사 완료. 독립 검토·실제 제출 미완료. 검증·holdout을 제외한 전체 학습 데이터로 새 학습. test 정답 미사용.\',encoding=\'utf-8\')\n    print(\'submission 생성 완료:\',final,flush=True);print(json.dumps(report,ensure_ascii=False,indent=2),flush=True)\n\n\n\ndef configure_pixels(adapter,pixels):\n    ip=adapter.processor.image_processor;ip.size={\'shortest_edge\':pixels,\'longest_edge\':pixels}\n    if hasattr(ip,\'min_pixels\'):ip.min_pixels=pixels\n    if hasattr(ip,\'max_pixels\'):ip.max_pixels=pixels\n\ndef stage_record(cfg,name):\n    return completed(Path(cfg[\'output_dir\']),name)\n\ndef commit(cfg,name,out,result):\n    files={str(p.relative_to(out)):sha(p) for p in out.rglob(\'*\') if p.is_file()}\n    write(Path(cfg[\'output_dir\'])/(name+\'_complete.json\'),{\'directory\':str(out),\'artifacts\':files,\'result\':result})\n\ndef split_data(cfg):\n    return load_split(cfg,Path(cfg[\'output_dir\']))\n\ndef prepare_colab(cfg):\n    import pandas as pd\n    from PIL import Image,ImageOps\n    root=Path(cfg[\'output_dir\']);test,sample=tables(cfg)\n    if sha(cfg[\'test_csv\'])!=cfg[\'test_sha256\'] or sha(cfg[\'sample_csv\'])!=cfg[\'sample_sha256\']:raise RuntimeError(\'입력 CSV 변경\')\n    import shutil,pandas as pd\n    base=Path(cfg[\'baseline_dir\'])\n    original=read(base/\'run_config.json\')\n    _,original_valid,_=RUNTIME_BASE.load_split(original,base)\n    old=stage_record(cfg,\'audit\')\n    if old:split_data(cfg)\n    else:\n        original_audit=RUNTIME_BASE.completed(base,\'audit\');src=Path(original_audit[\'directory\'])\n        out=root/(\'audit_\'+uuid.uuid4().hex[:8]);out.mkdir()\n        for name in [\'image_audit.csv\',\'data_findings.csv\',\'image_errors.csv\',\'near_duplicate_candidates.csv\']:\n            if (src/name).is_file():shutil.copy2(src/name,out/name)\n        split=pd.read_csv(src/\'split_manifest.csv\',dtype=str,keep_default_na=False)\n        if not split.split.isin([\'train\',\'valid\',\'holdout\',\'pool\']).all():raise ValueError(\'기존 split 이름 오류\')\n        preserved=split.loc[split.split.isin([\'valid\',\'holdout\'])].copy()\n        split.loc[split.split==\'pool\',\'split\']=\'train\'\n        if not split.loc[split.split.isin([\'valid\',\'holdout\'])].equals(preserved):raise RuntimeError(\'검증/holdout 변경\')\n        if split.groupby(\'group_id\').split.nunique().max()!=1:raise RuntimeError(\'이미지 그룹 분할 누수\')\n        split.to_csv(out/\'split_manifest.csv\',index=False)\n        for name in [\'train\',\'valid\',\'holdout\',\'pool\']:split[split.split==name].to_csv(out/f\'{name}_ids.csv\',index=False)\n        frame=pd.read_csv(Path(cfg[\'data_dir\'])/\'train.csv\',dtype=str,keep_default_na=False).merge(split,on=\'id\',validate=\'one_to_one\')\n        if len(frame)!=len(split) or set(frame.id)!=set(split.id):raise ValueError(\'train CSV와 분할 ID 불일치\')\n        pd.crosstab(frame.split,frame.answer).to_csv(out/\'answer_distribution.csv\');pd.crosstab(frame.split,frame.question_type).to_csv(out/\'type_distribution.csv\')\n        info=read(src/\'data_manifest.json\')\n        info.update(split_sha256=sha(out/\'split_manifest.csv\'),image_audit_sha256=sha(out/\'image_audit.csv\'),\n                    counts=split.split.value_counts().to_dict(),split_source=\'original fixed valid/holdout; original train+pool for training\')\n        if info[\'counts\'].get(\'valid\')!=500:raise RuntimeError(\'원래 검증이 500개인지 확인하세요.\')\n        write(out/\'data_manifest.json\',info);commit(cfg,\'audit\',out,info)\n    _,new_valid,_=split_data(cfg)\n    if new_valid.id.tolist()!=original_valid.id.tolist():raise RuntimeError(\'검증 ID/순서 변경\')\n    info=stage_record(cfg,\'audit\')[\'result\'];counts=info[\'counts\']\n    plan={\'counts\':counts,\'epochs\':1,\'optimizer_updates\':math.ceil(counts[\'train\']/cfg[\'gradient_accumulation\']),\n          \'reference_train_seconds_5060ti\':2729.7907537*counts[\'train\']/1000,\n          \'note\':\'linear estimate from local 1000-sample run; full training not yet measured\'}\n    write(root/\'training_plan.json\',plan);print(json.dumps(plan,ensure_ascii=False,indent=2),flush=True)\n    images=[]\n    for i,row in enumerate(test.to_dict(\'records\')):\n        p=resolve_image(cfg[\'data_dir\'],row[\'path\'])\n        with Image.open(p) as im:\n            rgb=ImageOps.exif_transpose(im).convert(\'RGB\');rgb.load();size=list(rgb.size)\n        images.append({\'id\':row[\'id\'],\'path\':row[\'path\'],\'sha256\':sha(p),\'size\':size})\n        if (i+1)%500==0:print(\'test 이미지 확인\',i+1,\'/\',len(test),flush=True)\n    frozen={\'fingerprint\':cfg[\'fingerprint\'],\'test_sha256\':sha(cfg[\'test_csv\']),\'sample_sha256\':sha(cfg[\'sample_csv\']),\n            \'images\':images,\'n\':len(test),\'revision\':cfg[\'revision\'],\'checkpoint\':\'pending\'}\n    path=root/\'frozen_inputs.json\'\n    if path.exists():\n        prior=read(path);frozen[\'checkpoint\']=prior[\'checkpoint\']\n        if frozen!=prior:raise RuntimeError(\'입력 변경. SESSION_TAG 변경 필요\')\n    write(path,frozen)\n    write(root/\'prompt.json\',{\'system\':SYSTEM_INSTRUCT,\'instruction\':cfg[\'p5_instruction\'],\'name\':\'P5\'})\n    print(\'입력 준비 완료:\',len(test),\'test 문항\',flush=True)\n\ndef train_colab(cfg):\n    import torch\n    root=Path(cfg[\'output_dir\'])\n    if stage_record(cfg,\'train\'):print(\'완료된 학습/모델 선택 재사용\',flush=True);return\n    tr,va,info=split_data(cfg);out=root/(\'train_\'+time.strftime(\'%Y%m%d_%H%M%S\')+\'_\'+uuid.uuid4().hex[:6]);out.mkdir()\n    gpu_environment(cfg,out)\n    install_answer_collator(RUNTIME_BASE)\n    adapter=ModelAdapter(cfg,out);adapter.add_lora();write(out/\'training_config.json\',cfg)\n    rng_cpu=torch.get_rng_state();rng_cuda=torch.cuda.get_rng_state_all()\n    audit_targets(sys.modules[__name__],adapter,tr,out)\n    # Small forward/backward check before optimizer training; reset RNG afterwards.\n    row=tr.iloc[0].to_dict();batch=adapter.encode(row,training=True);adapter.model.train()\n    with torch.autocast(\'cuda\',dtype=adapter.dtype):loss=adapter.loss(batch)\n    if not torch.isfinite(loss):raise FloatingPointError(\'smoke loss NaN/Inf\')\n    loss.backward();grads=[p.grad for p in adapter.model.parameters() if p.requires_grad and p.grad is not None]\n    if not grads or not all(bool(torch.isfinite(g).all()) for g in grads) or not any(bool(g.abs().max()>0) for g in grads):raise FloatingPointError(\'smoke gradient 오류\')\n    write(out/\'smoke.json\',{\'loss\':float(loss.detach()),\'finite_nonzero_gradients\':True,\'optimizer_step\':False})\n    adapter.model.zero_grad(set_to_none=True);del loss,batch,grads;gc.collect();torch.cuda.empty_cache()\n    # Baseline initialization consumes RNG before dropout. Preserve the post-initialization state below.\n    torch.set_rng_state(rng_cpu);torch.cuda.set_rng_state_all(rng_cuda)\n    result=train_one_epoch(adapter,tr,cfg,out)\n    checkpoint=out/\'adapter_epoch1\';adapter.model.save_pretrained(checkpoint);adapter.processor.save_pretrained(checkpoint)\n    configure_pixels(adapter,672**2)\n    before,_=evaluate(adapter,va.head(5),\'reload_before\',out);adapter.reload_adapter(checkpoint)\n    after,_=evaluate(adapter,va.head(5),\'reload_after\',out)\n    same=len(before)==len(after) and all(all(x[k]==y[k] for k in [\'id\',\'answer\',\'raw_output\']) for x,y in zip(before,after))\n    write(out/\'reload_check.json\',{\'identical_outputs\':same,\'n\':len(before)})\n    if not same:raise RuntimeError(\'저장/재로드 출력 불일치\')\n    result[\'checkpoint\']=str(checkpoint);commit(cfg,\'train\',out,result)\n\ndef validate_colab(cfg):\n    if cfg[\'mode\']==\'load_adapter\':print(\'업로드 모델은 새 검증 분할로 독립 Accuracy를 주장하지 않습니다.\',flush=True);return\n    if stage_record(cfg,\'validation\'):print(\'완료 검증 재사용\',flush=True);return\n    from peft import PeftModel\n    tr,va,info=split_data(cfg);root=Path(cfg[\'output_dir\']);rec=stage_record(cfg,\'train\')\n    if rec is None:raise RuntimeError(\'학습 먼저 실행\')\n    out=root/(\'validation_\'+uuid.uuid4().hex[:8]);out.mkdir();settings=dict(cfg);settings[\'pixel_budget\']=672**2\n    gpu_environment(settings,out);adapter=ModelAdapter(settings,out)\n    adapter.model=PeftModel.from_pretrained(adapter.model,rec[\'result\'][\'checkpoint\'],local_files_only=True,is_trainable=False)\n    adapter.generate(adapter.encode(va.iloc[0].to_dict(),training=False))\n    _,metrics=evaluate(adapter,va,\'valid\',out);commit(cfg,\'validation\',out,metrics)\n    print(\'전체 학습 검증 Accuracy:\',metrics[\'accuracy\'],flush=True)\n\ndef infer_colab(cfg):\n    import torch\n    from peft import PeftModel\n    root=Path(cfg[\'output_dir\']);f=read(root/\'frozen_inputs.json\');test,sample=tables(cfg)\n    if f[\'fingerprint\']!=cfg[\'fingerprint\'] or sha(cfg[\'test_csv\'])!=f[\'test_sha256\'] or sha(cfg[\'sample_csv\'])!=f[\'sample_sha256\']:raise RuntimeError(\'CSV 변경\')\n    for r in f[\'images\']:\n        if sha(resolve_image(cfg[\'data_dir\'],r[\'path\']))!=r[\'sha256\']:raise RuntimeError(\'test 이미지 변경\')\n    rec=stage_record(cfg,\'train\')\n    if rec is None:raise RuntimeError(\'학습/모델 선택 먼저 실행\')\n    checkpoint=rec[\'result\'][\'checkpoint\'];f[\'checkpoint\']=checkpoint;write(root/\'frozen_inputs.json\',f)\n    progress,records=load_progress(root,cfg[\'fingerprint\'],test.id.tolist())\n    if len(records)==len(test):print(\'전체 추론 완료 기록 재사용\',flush=True);return\n    out=root/(\'infer_\'+uuid.uuid4().hex[:8]);out.mkdir();settings=dict(cfg);settings[\'pixel_budget\']=672**2\n    gpu_environment(settings,out);adapter=ModelAdapter(settings,out)\n    adapter.model=PeftModel.from_pretrained(adapter.model,checkpoint,local_files_only=True,is_trainable=False)\n    ac=read(Path(checkpoint)/\'adapter_config.json\')\n    if ac[\'r\']!=8 or ac[\'lora_alpha\']!=16 or ac[\'lora_dropout\']!=.05:raise RuntimeError(\'LoRA 설정 불일치\')\n    adapter.generate(adapter.encode(test.iloc[len(records)].to_dict(),training=False));torch.cuda.synchronize()\n    torch.cuda.reset_peak_memory_stats();start=time.perf_counter();pending=[];before=len(records)\n    try:\n        for row in test.iloc[before:].to_dict(\'records\'):\n            tick=time.perf_counter();enc=adapter.encode(row,training=False);raw=adapter.generate(enc);torch.cuda.synchronize()\n            ans=parse_answer(raw) or \'\'\n            pending.append({\'id\':row[\'id\'],\'answer\':ans,\'raw_output\':raw,\'parse_failed\':not bool(ans),\'fallback_used\':False,\n                            \'seconds\':time.perf_counter()-tick,\'image_meta\':adapter.last_image_meta});del enc\n            if len(pending)>=cfg[\'save_every\']:\n                save_chunk(root,progress,pending);records.extend(pending);pending=[]\n                rate=(time.perf_counter()-start)/(len(records)-before)\n                print(f\'test {len(records)}/{len(test)} | {rate:.3f}초/문항 | 예상 잔여 {(len(test)-len(records))*rate/60:.1f}분\',flush=True)\n    finally:\n        save_chunk(root,progress,pending)\n        write(out/\'runtime.json\',{\'seconds\':time.perf_counter()-start,\'peak_allocated_gib\':torch.cuda.max_memory_allocated()/2**30,\'peak_reserved_gib\':torch.cuda.max_memory_reserved()/2**30})\n\n\nimport contextlib,importlib.util\ndef alive(pid):\n    if os.name==\'nt\':\n        import ctypes\n        from ctypes import wintypes\n        k=ctypes.WinDLL(\'kernel32\',use_last_error=True)\n        k.OpenProcess.argtypes=[wintypes.DWORD,wintypes.BOOL,wintypes.DWORD];k.OpenProcess.restype=wintypes.HANDLE\n        k.GetExitCodeProcess.argtypes=[wintypes.HANDLE,ctypes.POINTER(wintypes.DWORD)];k.CloseHandle.argtypes=[wintypes.HANDLE]\n        handle=k.OpenProcess(0x1000,False,pid)\n        if not handle:return False if ctypes.get_last_error()==87 else None\n        try:\n            code=wintypes.DWORD()\n            if not k.GetExitCodeProcess(handle,ctypes.byref(code)):return None\n            return code.value==259\n        finally:k.CloseHandle(handle)\n    try:os.kill(pid,0);return True\n    except ProcessLookupError:return False\n    except PermissionError:return None\n\n@contextlib.contextmanager\ndef lock_gpu(cfg):\n    lock=Path(cfg[\'project_dir\'])/\'output/TASK-006-local-alltrain/gpu_experiment.lock\';lock.parent.mkdir(parents=True,exist_ok=True)\n    for name in [\'TASK-006-lr\',\'TASK-006-loss\',\'TASK-006-prompts\',\'TASK-006-resolution\',\'TASK-006-training-resolution\']:\n        other=Path(cfg[\'project_dir\'])/\'output\'/name/\'gpu_experiment.lock\'\n        if other.exists():raise RuntimeError(f\'기존 실험 잠금: {other}. 다른 학습/추론이 없는지 확인하세요.\')\n    lr=read(Path(cfg[\'lr_dir\'])/\'resolution_config.json\')\n    if (Path(lr[\'baseline_dir\'])/\'running.lock\').exists():raise RuntimeError(\'baseline 실행 잠금이 남아 있습니다.\')\n    if lock.exists():\n        old=read(lock)\n        if alive(int(old[\'pid\'])) is False:lock.unlink()\n        else:raise RuntimeError(f\'제출 추론 프로세스 실행 중이거나 확인 불가: PID {old["pid"]}\')\n    token=uuid.uuid4().hex\n    fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    with os.fdopen(fd,\'w\') as f:json.dump({\'pid\':os.getpid(),\'token\':token},f)\n    try:yield\n    finally:\n        if lock.exists() and read(lock).get(\'token\')==token:lock.unlink()\n\n\n\ndef load_runtime(cfg):\n    global RUNTIME_BASE,ModelAdapter,train_one_epoch,evaluate,gpu_environment,parse_answer\n    for path,digest in cfg[\'source_hashes\'].items():\n        if sha(path)!=digest:raise RuntimeError(f\'기준 파일 변경: {path}\')\n    base=Path(cfg[\'baseline_dir\']);bc=read(base/\'run_config.json\')\n    worker=base/\'task006_worker.py\';data=worker.read_bytes()\n    if bc[\'worker_sha256\'] not in [hashlib.sha256(data).hexdigest(),hashlib.sha256(data.replace(b\'\\r\\n\',b\'\\n\')).hexdigest()]:raise RuntimeError(\'baseline 코드 내용 변경\')\n    spec=importlib.util.spec_from_file_location(\'actual_baseline_worker\',worker);m=importlib.util.module_from_spec(spec);sys.modules[spec.name]=m;spec.loader.exec_module(m)\n    m.block_network()\n    def p5(question,a,b,c,d):return f\'{question}\\n(a) {a}\\n(b) {b}\\n(c) {c}\\n(d) {d}\\n\\n\'+cfg[\'p5_instruction\']\n    m.build_mc_prompt=p5\n    RUNTIME_BASE=m;ModelAdapter=m.ModelAdapter;train_one_epoch=m.train_one_epoch;evaluate=m.evaluate;gpu_environment=m.gpu_environment;parse_answer=m.parse_answer\n    if cfg[\'mode\']!=\'train_fresh\' or cfg[\'train_n\']!=\'all_eligible\':raise ValueError(\'전체 새 학습 설정 필요\')\n    if cfg[\'pixel_budget\']!=640**2 or cfg[\'learning_rate\']!=1e-4 or cfg[\'loss_mode\']!=\'answer_only\' or cfg[\'prompt\']!=\'P5\':raise RuntimeError(\'고정 조건 불일치\')\n    if cfg[\'test_sha256\']!=sha(cfg[\'test_csv\']) or cfg[\'sample_sha256\']!=sha(cfg[\'sample_csv\']):raise RuntimeError(\'test/sample 변경\')\n    if cfg[\'split_sha256\']!=sha(cfg[\'split_manifest\']):raise RuntimeError(\'기존 분할 변경\')\n    rec=read(Path(cfg[\'lr_dir\'])/\'res_lr_1e-4_status.json\')\n    if rec[\'state\']!=\'completed\':raise RuntimeError(\'기준 LR 평가 미완료\')\n    for name,h in rec[\'artifacts\'].items():\n        if sha(Path(rec[\'directory\'])/name)!=h:raise RuntimeError(\'기준 평가 결과 변경\')\n\ndef comparison(cfg):\n    import pandas as pd\n    root=Path(cfg[\'output_dir\']);new=stage_record(cfg,\'validation\')\n    if not new:return\n    old=read(Path(cfg[\'lr_dir\'])/\'res_lr_1e-4_status.json\');old_dir=Path(old[\'directory\']);new_dir=Path(new[\'directory\'])\n    a=pd.read_csv(old_dir/\'valid_predictions.csv\',dtype=str,keep_default_na=False)\n    b=pd.read_csv(new_dir/\'valid_predictions.csv\',dtype=str,keep_default_na=False)\n    m=a.merge(b,on=\'id\',validate=\'one_to_one\',suffixes=(\'_1000\',\'_all\'))\n    if len(m)!=len(a) or len(m)!=len(b) or not (m.gold_1000==m.gold_all).all() or not (m.group_id_1000==m.group_id_all).all():raise RuntimeError(\'비교 문항 불일치\')\n    x=m.answer_1000==m.gold_1000;y=m.answer_all==m.gold_all\n    m[\'transition\']=[\'gain\' if v and not u else \'loss\' if u and not v else \'same_correct\' if u else \'same_wrong\' for u,v in zip(x,y)]\n    m.to_csv(root/\'paired_1000_vs_all.csv\',index=False,encoding=\'utf-8-sig\')\n    m[m.transition.isin([\'gain\',\'loss\'])].to_csv(root/\'changed_1000_vs_all.csv\',index=False,encoding=\'utf-8-sig\')\n    metrics=read(old_dir/\'valid_metrics.json\');training=stage_record(cfg,\'train\')[\'result\']\n    rows=[{\'condition\':\'previous_1000\',\'train_n\':cfg[\'reference_train_n\'],**metrics},\n          {\'condition\':\'all_eligible\',\'train_n\':training[\'train_n\'],**new[\'result\'],\'training_seconds\':training[\'seconds\'],\n           \'optimizer_updates\':training[\'updates\'],\'gain\':int((~x&y).sum()),\'loss\':int((x&~y).sum()),\'delta_pp\':100*float(y.mean()-x.mean())}]\n    for row in rows:row[\'accuracy_pct\']=100*row[\'accuracy\']\n    table=pd.DataFrame(rows);table.to_csv(root/\'data_size_comparison.csv\',index=False,encoding=\'utf-8-sig\')\n    print(table[[\'condition\',\'train_n\',\'correct_n\',\'accuracy_pct\',\'gain\',\'loss\',\'delta_pp\']].to_string(index=False),flush=True)\n\ndef run_local(cfg,stage):\n    root=Path(cfg[\'output_dir\']);root.mkdir(parents=True,exist_ok=True)\n    with lock_gpu(cfg):\n        load_runtime(cfg)\n        try:\n            if stage==\'prepare\':\n                assets=read(Path(cfg[\'baseline_dir\'])/\'model_assets.json\')\n                if assets[\'revision\']!=cfg[\'revision\']:raise RuntimeError(\'revision 변경\')\n                for name,item in assets[\'files\'].items():\n                    if sha(Path(cfg[\'model_dir\'])/name)!=item[\'sha256\']:raise RuntimeError(f\'모델 파일 변경: {name}\')\n            {\'prepare\':prepare_colab,\'train\':train_colab,\'validation\':validate_colab,\'infer\':infer_colab,\'export\':export}[stage](cfg)\n            if stage==\'validation\':comparison(cfg)\n            with open(root/\'CHANGELOG.md\',\'a\',encoding=\'utf-8\') as f:f.write(f\'\\n- {time.strftime("%Y-%m-%d %H:%M:%S")} {stage}: completed\\n\')\n        except BaseException as exc:write(root/(stage+\'_last_error.json\'),{\'type\':type(exc).__name__,\'message\':str(exc)});raise\n\nif __name__==\'__main__\':run_local(read(sys.argv[1]),sys.argv[2])\n'

In [5]:
CFG=dict(selected)
CFG.update(mode='train_fresh',train_n='all_eligible',output_dir='',run_dir='',
    project_dir=str(PROJECT_DIR),baseline_dir=str(BASELINE_RUN_DIR),lr_dir=str(LR_RUN_DIR),
    data_dir=str(DATA_DIR),model_dir=str(MODEL_DIR),test_csv=str(DATA_DIR/'test.csv'),sample_csv=str(DATA_DIR/'sample_submission.csv'),
    split_manifest=str(SPLIT_MANIFEST),split_sha256=file_hash(SPLIT_MANIFEST),
    test_sha256=file_hash(DATA_DIR/'test.csv'),sample_sha256=file_hash(DATA_DIR/'sample_submission.csv'),
    session_tag=SESSION_TAG,save_every=25,reference_train_n=train_record['training']['train_n'],
    worker_sha256=hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),environment_sha256=hashlib.sha256(freeze.encode()).hexdigest())
provenance=[LR_RUN_DIR/name for name in ['resolution_config.json','resolution_worker.py','train_lr_1e-4_status.json','res_lr_1e-4_status.json']]
provenance += [BASELINE_RUN_DIR/name for name in ['run_config.json','task006_worker.py','audit_complete.json']]
CFG['source_hashes']={str(p):file_hash(p) for p in provenance}
CFG['fingerprint']=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
OUTPUT_DIR=PROJECT_DIR/'output/TASK-006-local-alltrain'/('FULLTRAIN-'+CFG['fingerprint']);OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CFG['output_dir']=str(OUTPUT_DIR);CFG['run_dir']=str(OUTPUT_DIR)
WORKER=OUTPUT_DIR/'fulltrain_worker.py';CONFIG=OUTPUT_DIR/'run_config.json'
WORKER.write_bytes(WORKER_SOURCE.encode());CONFIG.write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'requirements.lock.txt').write_text(freeze,encoding='utf-8')

def run_stage(stage):
    process=None;env=os.environ.copy();env.update(PYTHONIOENCODING='utf-8',PYTHONUNBUFFERED='1',HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1')
    try:
        with open(OUTPUT_DIR/(stage+'.log'),'a',encoding='utf-8') as log:
            process=subprocess.Popen([str(ENV_PYTHON),'-u',str(WORKER),str(CONFIG),stage],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',env=env)
            for line in process.stdout:print(line,end='');log.write(line);log.flush()
            if process.wait()!=0:raise RuntimeError(f'{stage} 실패. {OUTPUT_DIR/(stage+".log")} 확인')
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
print('새 결과 폴더:',OUTPUT_DIR)


새 결과 폴더: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-local-alltrain\FULLTRAIN-032822484ddec52e


## 5. 기존 분할 확장
검증·holdout은 그대로 두고 pool을 train으로 합칩니다. training_plan.json에 실제 학습 수와 예상 step을 기록합니다.

In [6]:
run_stage("prepare")

{
  "counts": {
    "train": 5214,
    "holdout": 1000,
    "valid": 500
  },
  "epochs": 1,
  "optimizer_updates": 1304,
  "reference_train_seconds_5060ti": 14233.128989791801,
  "note": "linear estimate from local 1000-sample run; full training not yet measured"
}
test 이미지 확인 500 / 6714
test 이미지 확인 1000 / 6714
test 이미지 확인 1500 / 6714
test 이미지 확인 2000 / 6714
test 이미지 확인 2500 / 6714
test 이미지 확인 3000 / 6714
test 이미지 확인 3500 / 6714
test 이미지 확인 4000 / 6714
test 이미지 확인 4500 / 6714
test 이미지 확인 5000 / 6714
test 이미지 확인 5500 / 6714
test 이미지 확인 6000 / 6714
test 이미지 확인 6500 / 6714
입력 준비 완료: 6714 test 문항


## 6. 원본 모델에서 새 학습
선택 설정과 answer_only loss를 사용합니다. 소량 forward/backward 확인 후 학습하고 저장/재로드 출력을 검사합니다.

In [7]:
run_stage("train")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0923 09:49:53.975000 20672 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<09:43,  1.30it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

## 7. 고정 검증 500개 및 기존 1,000개 학습 모델과 비교
Accuracy·정답 수·개선/악화 문항·시간·VRAM을 저장합니다.

In [8]:
run_stage("validation")
p=OUTPUT_DIR/'validation_complete.json'
if p.exists():print(json.dumps(json.loads(p.read_text())['result'],ensure_ascii=False,indent=2))

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0923 13:47:59.333000 28036 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:49,  1.43it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

## 8. 전체 test 추론
RUN_TEST=True인 경우 실행합니다. 저장된 문항 다음부터 재개합니다.

In [9]:
if RUN_TEST:run_stage("infer")
else:print("전체 test 추론 생략")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0923 13:55:27.491000 29176 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:48,  1.44it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

## 9. submission.csv 생성
RUN_TEST=True인 경우 실행합니다. 파싱 실패·누락을 임의 답으로 채우지 않습니다. 자동 제출하지 않습니다.

In [10]:
if RUN_TEST:
    run_stage("export")
    from IPython.display import display,FileLink
    print('제출 CSV:',OUTPUT_DIR/'submission.csv')
    display(FileLink(str(OUTPUT_DIR/'submission.csv')))
else:print('학습량 실험만 완료. data_size_comparison.csv를 확인하세요.')


submission 생성 완료: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-local-alltrain\FULLTRAIN-032822484ddec52e\submission.csv
{
  "n": 6714,
  "parse_failure_n": 0,
  "fallback_n": 0,
  "id_order_matches_sample": true,
  "checkpoint": "C:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\output\\TASK-006-local-alltrain\\FULLTRAIN-032822484ddec52e\\train_20260923_094943_eef39a\\adapter_epoch1",
  "auto_submitted": false,
  "review_02": "pending",
  "test_accuracy": null,
  "state": "file_checks_passed",
  "sha256": "a3a51c0e74ff33b087e790d57ad7eb2a5e9cf14fc0475a12d8ffbf2ecb5244e3",
  "path": "C:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\output\\TASK-006-local-alltrain\\FULLTRAIN-032822484ddec52e\\submission.csv",
  "answer_counts": {
    "a": 1704,
    "d": 1704,
    "c": 1658,
    "b": 1648
  }
}
제출 CSV: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-local-alltrain\FULLTRAIN-032822484ddec52e\submission.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-local-alltrain\FULLTRAIN-032822484ddec52e\submission.csv

## 확인할 파일

| 파일 | 내용 |
|---|---|
| data_size_comparison.csv | 기존 1,000개 vs 전체 학습, Accuracy·개선/악화·시간 |
| paired_1000_vs_all.csv / changed_1000_vs_all.csv | 문항별 정오 전환 및 변경 문항 |
| training_plan.json | 실제 학습 개수·optimizer step·시간 참고치 |
| audit_*/split_manifest.csv | 고정 검증·holdout과 확대된 train ID |
| train_*/adapter_epoch1/ | 새 LoRA와 processor |
| train_*/training_config.json / train_log.csv / supervised_tokens.csv | 학습 설정·loss 추이·감독 토큰 |
| validation_*/valid_metrics.json / valid_predictions.csv | 새 모델 검증 지표·예측 |
| submission.csv / submission_validation.json | 제출 파일과 ID·형식·답변 검사 |
| test_predictions.csv / parse_failures.csv / chunks/ | test 출력·실패·부분 재개 기록 |

원래 1,000개보다 성능이 좋아진다는 보장은 없습니다. 같은 검증셋에서 개선/악화와 비용을 함께 비교하세요.
분할 확대·최고 모델 교체·최종 제출 검토는 별도이며 자동 채택·자동 제출하지 않습니다.
작성 단계에서는 문법과 CPU 로직을 검사했습니다. 실제 GPU 전체 학습은 미검증입니다.
